# 16.4 - GenAI in Software Engineering: The Model Drafts, You Verify

**Module 16 - Industry Applications** | Netsetos GenAI Engineering

This notebook builds the harness that makes AI-generated code safe to ship. Across seven small, keyless checks you assemble a test gate, a bounded agent loop, a context selector, a severity-ranked review, a branch-coverage check, a CI merge gate, and an honest productivity metric - and you watch one off-by-one bug get caught by six of them in a row.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

This notebook is deterministic - no random draws, no model calls, no external packages. This first cell is just a note that the examples use fixed, seeded values so every run prints the same thing.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A one-line reproducibility marker, not executable logic. Everything downstream is plain Python running on hardcoded illustrative data, so the outputs are stable and keyless.

**How the code works - step by step**
- A single comment line noting the lesson uses fixed values throughout, so results are reproducible.
- No imports, no pip install - the whole notebook runs on the standard library.

**In one line:** deterministic by design - every cell prints the same output every run.

**What you'll see:** (no output - environment prep)

## 1 - Generate, then verify

An AI writes a plausible function in seconds, but a draft that *reads* correctly is not the same as one that *is* correct. The discipline of this whole lesson starts here: a generated function is a guess until its tests are green. This cell runs the tests as the gate that decides accept or reject.

In [ ]:
# GENERATE, THEN VERIFY: an AI writes a function draft in seconds, but a draft is not done code - it is done when the tests are
# green. Never trust unverified generation. Here the model output is illustrative; the runnable part is the test-suite GATE.
def is_palindrome(s):                                     # pretend this was generated from the spec "ignore case and non-letters"
    clean = [c.lower() for c in s if c.isalpha()]
    return clean == clean[::-1]
TESTS = [                                                  # the spec, encoded as tests: (input, expected)
    ("racecar", True), ("RaceCar", True), ("A man, a plan, a canal: Panama", True),
    ("hello", False), ("", True)]
results = [(inp, is_palindrome(inp), exp) for inp, exp in TESTS]
passed = [r for r in results if r[1] == r[2]]
print("Spec -> generated draft -> run the tests (the gate):")
for inp, got, exp in results:
    ok = got == exp
    print("  [{}] is_palindrome({!r}) = {}  (expected {})".format("PASS" if ok else "FAIL", inp[:22], got, exp))
print()
print("{}/{} tests pass -> {}".format(len(passed), len(TESTS),
      "ACCEPT the generated code" if len(passed) == len(TESTS) else "REJECT - send the failures back to the model"))
print("A generated function that has not passed its tests is a guess, not code. The tests are the contract the draft must clear.")

# Output:
# Spec -> generated draft -> run the tests (the gate):
#   [PASS] is_palindrome('racecar') = True  (expected True)
#   [PASS] is_palindrome('RaceCar') = True  (expected True)
#   [PASS] is_palindrome('A man, a plan, a canal') = True  (expected True)
#   [PASS] is_palindrome('hello') = False  (expected False)
#   [PASS] is_palindrome('') = True  (expected True)
#
# 5/5 tests pass -> ACCEPT the generated code
# A generated function that has not passed its tests is a guess, not code. The tests are the contract the draft must clear.

**Explanation**

This is a test-suite gate, not a model call - the function is shown illustratively and the runnable part is the check. It encodes a spec ("ignore case and punctuation") as five input/expected pairs, runs the draft against all of them, and turns the pass count into an accept-or-reject verdict. Read it as: the tests are the contract, and only a green suite lets a draft count as code.

**How the code works - step by step**
- **`is_palindrome(s)`** - the pretend-generated draft: lowercases the letters, drops non-alpha characters, and checks the cleaned list against its reverse.
- **`TESTS`** - the spec encoded as five `(input, expected)` pairs, including the empty string and a punctuation-heavy sentence.
- **`results` / `passed`** - runs the draft on each input and keeps the rows where the actual output matches the expected.
- **The verdict line** - if all five pass, print ACCEPT; if any fail, print REJECT and send the failures back to the model.

**In one line:** encode the spec as tests, run the draft against them, accept only on all-green.

**What you'll see:** A five-row PASS/FAIL table (all PASS here), then "5/5 tests pass -> ACCEPT the generated code" and a reminder that an untested draft is a guess, not code.

## 2 - The bounded agent loop

Rejecting a failing draft only helps if something fixes it. That something is the coding-agent loop - plan, code, test, fix, repeat - feeding the real test errors back each round. But a stuck agent will retry forever, so every real loop carries a hard iteration cap. This cell runs the loop with a cap.

In [ ]:
# THE CODING-AGENT LOOP: plan -> edit -> run tests -> fix -> repeat, with a HARD iteration cap so a stuck agent cannot loop
# forever. The loop is the pattern behind Claude Code, Aider, and Cursor; the cap is what keeps it from burning your budget.
MAX_ITERS = 5
# a deterministic model of the fix loop: tests failing this round, and how many the next fix resolves
failing_by_round = [3, 2, 1, 0]                            # illustrative: the agent fixes some failures each iteration
def agent_loop(failing_by_round, cap):
    for i, failing in enumerate(failing_by_round):
        if i >= cap:
            return {"status": "CAPPED", "iterations": i, "failing": failing}
        print("  iter {}: run tests -> {} failing".format(i + 1, failing))
        if failing == 0:
            return {"status": "DONE", "iterations": i + 1, "failing": 0}
        print("           {} still failing -> feed the errors back, fix, re-run".format(failing))
    return {"status": "CAPPED", "iterations": len(failing_by_round), "failing": failing_by_round[-1]}
print("Coding agent: plan -> code -> test -> fix, capped at {} iterations".format(MAX_ITERS))
r = agent_loop(failing_by_round, MAX_ITERS)
print()
print("result: {} after {} iterations, {} tests failing".format(r["status"], r["iterations"], r["failing"]))
print("The loop converges because every round feeds the real test errors back to the model. The cap is the safety valve:")
print("without it, an agent that cannot fix the last failure re-tries forever - a bounded loop is the difference from a runaway.")

# Output:
# Coding agent: plan -> code -> test -> fix, capped at 5 iterations
#   iter 1: run tests -> 3 failing
#            3 still failing -> feed the errors back, fix, re-run
#   iter 2: run tests -> 2 failing
#            2 still failing -> feed the errors back, fix, re-run
#   iter 3: run tests -> 1 failing
#            1 still failing -> feed the errors back, fix, re-run
#   iter 4: run tests -> 0 failing
#
# result: DONE after 4 iterations, 0 tests failing
# The loop converges because every round feeds the real test errors back to the model. The cap is the safety valve:
# without it, an agent that cannot fix the last failure re-tries forever - a bounded loop is the difference from a runaway.

**Explanation**

A deterministic simulation of the plan-code-test-fix loop, driven by a hardcoded schedule of failing-test counts rather than a live model. It shows two exits from one function: converging to DONE when failures reach zero, and stopping at CAPPED when the cap is hit. The point is the cap - the safety valve that separates a converging agent from a runaway that burns your budget.

**How the code works - step by step**
- **`MAX_ITERS = 5`** - the hard iteration cap.
- **`failing_by_round = [3, 2, 1, 0]`** - an illustrative schedule: the agent resolves some failures each round until none remain.
- **`agent_loop(failing_by_round, cap)`** - iterates the rounds; if the index reaches the cap it returns CAPPED, if the failing count hits 0 it returns DONE, otherwise it prints "feed the errors back, fix, re-run" and continues.
- **The result line** - reports status, iteration count, and remaining failures.

**In one line:** correct against real test errors each round, but stop at the cap so a stuck agent cannot loop forever.

**What you'll see:** The failing count printed falling 3 -> 2 -> 1 -> 0 across four iterations, then "result: DONE after 4 iterations, 0 tests failing," plus a note that the cap is the safety valve against a runaway.

## 3 - Context selection

The loop can only edit what it can see, and it cannot see the whole repo - a real codebase is far bigger than any context window. So the agent must rank files by relevance to the task and pack the most relevant ones into a token budget. This cell ranks and packs; watch a relevant file get dropped purely on size.

In [ ]:
# CONTEXT SELECTION: an agent cannot read the whole repo - it has a token budget. Rank files by relevance to the task and pack
# the most relevant ones that FIT. Bad context is the top cause of bad AI edits; the ranking is the retrieval step for code.
TASK = "fix the JWT token expiry bug in auth"
BUDGET = 6000; FLOOR = 0.5                                # token budget + a minimum relevance to bother packing a file
FILES = [  # (path, relevance score 0-1, token cost)
    ("auth/jwt.py",        0.95, 1800),
    ("auth/middleware.py", 0.70, 2600),
    ("auth/tests.py",      0.60, 3200),
    ("users/models.py",    0.30, 1500),
    ("billing/stripe.py",  0.05, 2400)]
ranked = sorted(FILES, key=lambda f: f[1], reverse=True)
picked, used, reason = [], 0, {}
for path, rel, cost in ranked:
    if rel < FLOOR:
        reason[path] = "below the {:.1f} relevance floor".format(FLOOR)
    elif used + cost <= BUDGET:
        picked.append(path); used += cost
    else:
        reason[path] = "relevant, but {} tok does not fit the remaining budget".format(cost)
print("Task: {!r}  |  budget: {} tokens, relevance floor {}".format(TASK, BUDGET, FLOOR))
print("Ranked by relevance, packed while each fits the budget:")
for path, rel, cost in ranked:
    mark = "PICK" if path in picked else "skip"
    tail = "" if path in picked else "  <- " + reason[path]
    print("  [{}] {:<20} rel={:.2f} cost={}{}".format(mark, path, rel, cost, tail))
print()
print("selected {} files, {}/{} tokens used.".format(len(picked), used, BUDGET))
print("Note the nuance: auth/tests.py clears the relevance floor (0.60) but does NOT fit the remaining budget after the two")
print("higher-ranked files - a relevant file can still be dropped by size. billing and the users model fall below the floor.")
print("Choosing the right context under a hard budget is most of what makes an AI code edit land.")

# Output:
# Task: 'fix the JWT token expiry bug in auth'  |  budget: 6000 tokens, relevance floor 0.5
# Ranked by relevance, packed while each fits the budget:
#   [PICK] auth/jwt.py          rel=0.95 cost=1800
#   [PICK] auth/middleware.py   rel=0.70 cost=2600
#   [skip] auth/tests.py        rel=0.60 cost=3200  <- relevant, but 3200 tok does not fit the remaining budget
#   [skip] users/models.py      rel=0.30 cost=1500  <- below the 0.5 relevance floor
#   [skip] billing/stripe.py    rel=0.05 cost=2400  <- below the 0.5 relevance floor
#
# selected 2 files, 4400/6000 tokens used.
# Note the nuance: auth/tests.py clears the relevance floor (0.60) but does NOT fit the remaining budget after the two
# higher-ranked files - a relevant file can still be dropped by size. billing and the users model fall below the floor.
# Choosing the right context under a hard budget is most of what makes an AI code edit land.

**Explanation**

This is retrieval applied to code: a ranking-and-packing routine over a small file table, no model involved. It sorts candidate files by a relevance score, then greedily packs them while each clears a relevance floor and still fits the remaining token budget. The subtle lesson is that selection is ranking *and* fitting - a relevant file can be dropped because it is too big, and an irrelevant file that crowds out the buggy one is worse than empty context.

**How the code works - step by step**
- **`TASK`, `BUDGET`, `FLOOR`** - the task string, a 6000-token budget, and a 0.5 minimum relevance to bother packing a file.
- **`FILES`** - five `(path, relevance, token_cost)` rows spanning auth, users, and billing.
- **`ranked`** - sorts the files by relevance, highest first.
- **The packing loop** - for each ranked file: skip if below the floor, else pack it if it fits the remaining budget, else record it as relevant-but-too-big.
- **The report** - prints each file as PICK or skip with the skip reason, then the total tokens used.

**In one line:** rank by relevance, then pack only the files that clear the floor and fit the budget.

**What you'll see:** auth/jwt.py and auth/middleware.py picked (4400/6000 tokens); auth/tests.py skipped because it clears the floor but does not fit; users/models.py and billing/stripe.py skipped below the 0.5 floor - with a note that a relevant file can still be dropped by size.

## 4 - AI code review

The agent proposes a change; now something has to judge it. AI code review returns findings tagged by severity, and the defining rule is that *severity, not count,* drives the verdict: one HIGH correctness or security finding forces request-changes no matter how many nits are clean. This cell runs the severity-ranked review - and it catches the cold open's off-by-one.

In [ ]:
# AI CODE REVIEW: a diff comes back as findings ranked by SEVERITY, and the severity decides the verdict. A single HIGH finding
# (a security or correctness bug) means request-changes, no matter how many nits are clean. AI flags; a human still decides.
findings = [  # (file, severity, note) - illustrative output of a review pass over a diff
    ("auth/jwt.py",   "high", "expiry compared with '<' not '<=' - token valid one second past expiry"),
    ("auth/jwt.py",   "med",  "no test covers the exactly-at-expiry boundary"),
    ("auth/util.py",  "low",  "variable name 'x' is unclear")]
order = {"high": 0, "med": 1, "low": 2}
findings.sort(key=lambda f: order[f[1]])
counts = {s: sum(1 for f in findings if f[1] == s) for s in ("high", "med", "low")}
verdict = "REQUEST CHANGES" if counts["high"] else ("COMMENT" if counts["med"] else "APPROVE")
print("AI review of the diff ({} findings), most severe first:".format(len(findings)))
for f, sev, note in findings:
    print("  [{:<4}] {}: {}".format(sev.upper(), f, note))
print()
print("counts: {} high, {} med, {} low  ->  verdict: {}".format(counts["high"], counts["med"], counts["low"], verdict))
print("Severity, not count, drives the verdict: one HIGH correctness bug blocks the merge while three style nits would not.")
print("The AI surfaces and ranks; the human reads the HIGH finding and decides. AI flags, a person approves.")

# Output:
# AI review of the diff (3 findings), most severe first:
#   [HIGH] auth/jwt.py: expiry compared with '<' not '<=' - token valid one second past expiry
#   [MED ] auth/jwt.py: no test covers the exactly-at-expiry boundary
#   [LOW ] auth/util.py: variable name 'x' is unclear
#
# counts: 1 high, 1 med, 1 low  ->  verdict: REQUEST CHANGES
# Severity, not count, drives the verdict: one HIGH correctness bug blocks the merge while three style nits would not.
# The AI surfaces and ranks; the human reads the HIGH finding and decides. AI flags, a person approves.

**Explanation**

A verdict function over a list of findings, illustrative review output rather than a live pass. It sorts findings by a severity rank, counts each level, and maps the presence of any HIGH to REQUEST CHANGES. The design point is that the AI surfaces and ranks the findings, but a human reads the HIGH one and decides - AI flags, a person approves.

**How the code works - step by step**
- **`findings`** - three illustrative `(file, severity, note)` rows: a HIGH (`<` instead of `<=` on expiry - the cold open's bug), a MED (no boundary test), and a LOW (unclear variable name).
- **`order` / `findings.sort`** - ranks severities high < med < low and sorts most-severe first.
- **`counts`** - tallies how many findings sit at each severity.
- **`verdict`** - REQUEST CHANGES if any HIGH, else COMMENT if any MED, else APPROVE.

**In one line:** rank findings by severity and let a single HIGH block the merge, regardless of the nit count.

**What you'll see:** The three findings printed most-severe first, then "counts: 1 high, 1 med, 1 low -> verdict: REQUEST CHANGES," with the reminder that severity, not count, drives the verdict and a human makes the call.

## 5 - Test coverage

Tests only gate the code if they actually exercise it, and generated tests have a habit of covering the obvious paths while skipping the awkward one. So you measure branch coverage and flag the gap - because the untested branch is exactly where the AI's bug hides. This cell measures coverage, and the uncovered branch is the same one the review flagged HIGH.

In [ ]:
# TEST GENERATION + COVERAGE: generated tests must actually EXERCISE the code, not just exist. Measure branch coverage and flag
# the gap - an untested branch is where the AI's bug hides. "The AI wrote tests" is worthless if they skip the risky path.
BRANCHES = ["empty input", "valid token", "expired token", "malformed token", "at-expiry boundary"]
covered_by_tests = {"empty input", "valid token", "expired token", "malformed token"}   # the AI's tests skipped one
covered = [b for b in BRANCHES if b in covered_by_tests]
missing = [b for b in BRANCHES if b not in covered_by_tests]
pct = round(100 * len(covered) / len(BRANCHES))
print("Branch coverage of the generated tests ({} branches):".format(len(BRANCHES)))
for b in BRANCHES:
    print("  [{}] {}".format("x" if b in covered_by_tests else " ", b))
print()
print("coverage: {}/{} branches = {} percent".format(len(covered), len(BRANCHES), pct))
print("uncovered: {}".format(", ".join(missing)))
print("The one skipped branch - the at-expiry boundary - is exactly the HIGH bug the review found. Generated tests that miss")
print("the risky branch give false confidence: measure coverage and demand the gap be closed, do not just count that tests exist.")

# Output:
# Branch coverage of the generated tests (5 branches):
#   [x] empty input
#   [x] valid token
#   [x] expired token
#   [x] malformed token
#   [ ] at-expiry boundary
#
# coverage: 4/5 branches = 80 percent
# uncovered: at-expiry boundary
# The one skipped branch - the at-expiry boundary - is exactly the HIGH bug the review found. Generated tests that miss
# the risky branch give false confidence: measure coverage and demand the gap be closed, do not just count that tests exist.

**Explanation**

A coverage calculator over a fixed set of branches, no test runner needed. It compares the full branch list against the set the generated tests actually touch, computes a percentage, and names the uncovered branch. The lesson is that the risky branch and the untested branch are the same branch - a suite that skips it gives false confidence, so "the AI wrote tests" is worthless if the tests avoid the path that fails.

**How the code works - step by step**
- **`BRANCHES`** - the five branches of the function, including the `at-expiry boundary`.
- **`covered_by_tests`** - the four branches the generated tests actually hit; the boundary is missing.
- **`covered` / `missing` / `pct`** - splits the branches into hit and unhit and computes the coverage percentage.
- **The checklist print** - marks each branch `[x]` or `[ ]`, then reports coverage and names the uncovered branch.

**In one line:** measure which branches the tests exercise and surface the one they skip.

**What you'll see:** A five-branch checklist with `at-expiry boundary` unchecked, then "coverage: 4/5 branches = 80 percent" and "uncovered: at-expiry boundary" - the exact branch the review flagged HIGH.

## 6 - The CI merge gate

Every guardrail so far converges on one decision: does this change merge to main? The CI merge gate makes that call, and its defining rule is that AI-written code goes through the *same* gate as human code - lint, tests, coverage, review with no HIGH open, and a human approval - and it never auto-merges. This cell runs the gate; one unmet check blocks the merge.

In [ ]:
# THE CI MERGE GATE: AI-written code goes through the SAME gate as human code - lint, tests, review, and a human approval - and
# NEVER auto-merges. One unmet check blocks the merge. The gate is where "the AI wrote it" stops mattering and "it passed" starts.
GATE = {                                                  # each must hold to merge
    "lint clean":                 True,
    "all tests pass":             True,
    "branch coverage >= target":  False,   # the uncovered at-expiry branch (from the coverage step)
    "code review: no HIGH open":  False,   # the HIGH correctness finding is still open
    "a human approved the merge": False}   # never auto-merge from an AI
unmet = [k for k, ok in GATE.items() if not ok]
print("CI MERGE GATE (same gate for AI and human code):")
for k, ok in GATE.items():
    print("  [{}] {}".format("x" if ok else " ", k))
print()
print("decision: {}".format("MERGE" if not unmet else "BLOCK - {} unmet:".format(len(unmet))))
for u in unmet:
    print("   - " + u)
print("AI code is not special-cased: it clears lint, tests, coverage, and review, and a human approves - or it does not merge.")
print("Never auto-merge from an AI. The gate is what lets you move fast on drafts without shipping an unreviewed bug to main.")

# Output:
# CI MERGE GATE (same gate for AI and human code):
#   [x] lint clean
#   [x] all tests pass
#   [ ] branch coverage >= target
#   [ ] code review: no HIGH open
#   [ ] a human approved the merge
#
# decision: BLOCK - 3 unmet:
#    - branch coverage >= target
#    - code review: no HIGH open
#    - a human approved the merge
# AI code is not special-cased: it clears lint, tests, coverage, and review, and a human approves - or it does not merge.
# Never auto-merge from an AI. The gate is what lets you move fast on drafts without shipping an unreviewed bug to main.

**Explanation**

A boolean AND-gate over a dictionary of named checks, evaluated as a single merge-or-block decision. It collects every unmet check and blocks if any remain, treating AI code with no special case or fast lane. The single most important rule encoded here is the last check: a human must approve - never auto-merge from an AI.

**How the code works - step by step**
- **`GATE`** - five named checks mapped to booleans: lint and tests pass (True), while coverage, no-HIGH-open, and human-approval are still False.
- **`unmet`** - collects the keys whose check is False.
- **The checklist print** - marks each check `[x]` or `[ ]`.
- **The decision line** - MERGE only if `unmet` is empty, otherwise BLOCK and list every unmet check.

**In one line:** the same five checks for AI and human code, and any single failure blocks the merge.

**What you'll see:** lint and tests checked, coverage/no-HIGH/human-approval unchecked, then "decision: BLOCK - 3 unmet" listing the three, and the rule: never auto-merge from an AI.

## 7 - The honest metric

The harness is built; the last discipline is measuring whether it actually helped - honestly. "3-5x faster" is a feeling and lines-of-code is a vanity number, because a model can emit ten thousand lines you throw away. This cell computes the metrics that mean something: acceptance and churn.

In [ ]:
# THE HONEST METRIC: "3-5x faster" is a vanity claim. Measure AI-assisted engineering by ACCEPTANCE (how much suggested code
# survives review) and CHURN (how much shipped code is rewritten soon after). Lines generated is calories eaten, not fitness.
suggested_lines = 1000
accepted_lines  = 620      # survived review and made it into a PR
reverted_lines  = 130      # shipped, then rewritten or reverted within two weeks (rework)
acceptance = round(100 * accepted_lines / suggested_lines)
churn      = round(100 * reverted_lines / accepted_lines)
net_kept   = accepted_lines - reverted_lines
print("AI-assisted engineering, measured honestly:")
print("  suggested lines: {}".format(suggested_lines))
print("  accepted (survived review): {}  -> acceptance {} percent".format(accepted_lines, acceptance))
print("  reverted/rewritten soon after: {}  -> churn {} percent of accepted".format(reverted_lines, churn))
print("  net durable lines kept: {}".format(net_kept))
print()
print("Lines generated says nothing - a model can emit 10,000 lines of code you throw away. Acceptance and churn measure whether")
print("the assistance actually helped: high acceptance with low churn is real leverage; high churn means you paid the speed back")
print("in rework. Track the durable outcome, not the raw output, or you optimize for a number that does not ship value.")

# Output:
# AI-assisted engineering, measured honestly:
#   suggested lines: 1000
#   accepted (survived review): 620  -> acceptance 62 percent
#   reverted/rewritten soon after: 130  -> churn 21 percent of accepted
#   net durable lines kept: 490
#
# Lines generated says nothing - a model can emit 10,000 lines of code you throw away. Acceptance and churn measure whether
# the assistance actually helped: high acceptance with low churn is real leverage; high churn means you paid the speed back
# in rework. Track the durable outcome, not the raw output, or you optimize for a number that does not ship value.

**Explanation**

A tiny arithmetic cell that turns three raw line counts into two ratios plus a durable total. It computes acceptance (share of suggested code that survives review) and churn (share of accepted code reverted or rewritten soon after), then the net durable lines kept. The message is that lines emitted is calories eaten, not fitness - high acceptance with low churn is real leverage, while high churn means the speed was paid straight back in rework.

**How the code works - step by step**
- **`suggested_lines`, `accepted_lines`, `reverted_lines`** - 1000 suggested, 620 that survived review, 130 reverted or rewritten within two weeks.
- **`acceptance`** - accepted over suggested, as a percentage.
- **`churn`** - reverted over accepted, as a percentage of what was accepted.
- **`net_kept`** - accepted minus reverted: the lines that actually lasted.

**In one line:** measure durable, accepted code - acceptance and churn - not raw lines generated.

**What you'll see:** "acceptance 62 percent," "churn 21 percent of accepted," and "net durable lines kept: 490," with a note that lines generated is a vanity metric - durable accepted code, not raw output, is the real leverage.

The through-line is a single one-character bug - `<` where `<=` was meant - that reads as clean code and slips past a skim. You watched it get caught six ways: the test gate rejects the draft, the review flags it HIGH, the coverage check shows its branch is untested, and the merge gate blocks on it. The lesson in one sentence: an AI does not remove the engineering, it moves the engineer's job from writing the draft to verifying it. Lesson 16.5 carries the same guardrails to non-developers building with no-code and low-code tools.